# SchemaShift on Colab: many open models

Runs a list of open models on the SchemaShift tasks with a free Colab GPU, using the same
`scripts/run_models.py` as on the laptop. Each model is run twice: without and with the
one-line hint.

1. **Runtime -> Change runtime type -> T4 GPU.**
2. Run the cells in order. Step 2 asks for `schemashift_student_club.zip` (scripts, the four
   changed databases, `tasks.jsonl` and the results so far).
3. Step 3 (Google Drive) is strongly recommended: free Colab stops after a few hours, and Drive
   keeps every finished run. Next time, run the same cells again and the loop continues where
   it stopped.
4. Step 8 prints the comparison table, step 9 downloads `results.zip`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload the project zip and unpack it

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # choose schemashift_student_club.zip
zipfile.ZipFile(next(iter(uploaded))).extractall("/content/Text_To_Sql")
%cd /content/Text_To_Sql
!ls scripts results data/schemashift/student_club

## 3. Keep results in Google Drive (recommended)

Saves results straight into `My Drive/SchemaShift/results`. Results already in the zip are
copied there first, without overwriting anything.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/SchemaShift/results
!cp -n results/* /content/drive/MyDrive/SchemaShift/results/ 2>/dev/null; true
!rm -rf results && ln -s /content/drive/MyDrive/SchemaShift/results results
!ls results

## 4. Install the Python package

In [ ]:
!pip -q install sqlglot==30.23.0

## 5. Install Ollama and start it

The context window is set to 8,192 tokens. With a smaller window, Ollama cuts the start of a
long prompt (the instructions and the schema) without any warning.

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time, urllib.request

server = subprocess.Popen(["ollama", "serve"], env=dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192"),
                          stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
!ollama --version

## 6. Choose the models

Put `#` in front of a line to skip that model. The list is in priority order, so the most
important comparisons finish first. Rough time per run on a T4: 1.5-8B about 5-10 minutes,
12-16B about 15-20 minutes, reasoning models much longer because they think before answering.

In [ ]:
MODELS = [
    # 1) Size ladder: same family, only the size changes (7b is already done)
    {"name": "qwen2.5-coder:1.5b"},
    {"name": "qwen2.5-coder:3b"},
    {"name": "qwen2.5-coder:7b"},
    {"name": "qwen2.5-coder:14b"},
    # 2) SQL specialists from Hugging Face (trained to write SQL from a question)
    {"name": "hf.co/mradermacher/OmniSQL-7B-GGUF:Q4_K_M"},
    {"name": "hf.co/mradermacher/Arctic-Text2SQL-R1-7B-GGUF:Q4_K_M", "max_tokens": 4096},
    # {"name": "hf.co/mradermacher/XiYanSQL-QwenCoder-7B-2504-GGUF:Q4_K_M"},
    # 3) Other model families
    {"name": "llama3.1:8b"},
    {"name": "gemma3:12b"},
    {"name": "deepseek-coder-v2:16b"},
    {"name": "phi4:14b"},
    # 4) Reasoning models: slow, and they need room for their thinking
    {"name": "qwen3:8b", "max_tokens": 4096},
    {"name": "deepseek-r1:8b", "max_tokens": 4096},
]

## 7. Run every model (safe to stop and run again)

For each model: skip it if both runs are already finished, otherwise download it, run without
and with the hint, then delete it to free disk space. Every finished task is saved at once,
so a stopped session loses nothing.

In [ ]:
import datetime, json, subprocess, sys
sys.path.insert(0, "scripts")
from run_models import run_name

TASK_DIR = "data/schemashift/student_club"
N_TASKS = sum(1 for _ in open(f"{TASK_DIR}/tasks.jsonl", encoding="utf-8"))
OLLAMA_VERSION = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()


def finished(model, hint):
    path = f"results/{run_name('student_club', model, 'adapt', hint)}.jsonl"
    try:
        return sum(1 for _ in open(path, encoding="utf-8")) >= N_TASKS
    except FileNotFoundError:
        return False


def stream(cmd):
    """Run a command and show its output live; returns the exit code."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


for m in MODELS:
    name, max_tokens = m["name"], m.get("max_tokens", 1024)
    if finished(name, False) and finished(name, True):
        print(f"== {name}: already done, skipping")
        continue
    print(f"\n========== {name} ==========\ndownloading ...")
    pull = subprocess.run(["ollama", "pull", name], capture_output=True, text=True)
    if pull.returncode != 0:
        print(f"!! could not download {name}: {pull.stderr.strip()[-300:]}")
        continue
    for hint in (False, True):
        if finished(name, hint):
            continue
        cmd = [sys.executable, "scripts/run_models.py", TASK_DIR, "--model", name,
               "--max-tokens", str(max_tokens)] + (["--hint"] if hint else [])
        code = stream(cmd)
        with open("results/run_log.jsonl", "a", encoding="utf-8") as log:
            log.write(json.dumps({
                "model": name, "hint": hint, "exit_code": code, "finished": finished(name, hint),
                "ollama": OLLAMA_VERSION,
                "at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            }) + "\n")
    subprocess.run(["ollama", "rm", name], capture_output=True)  # free disk for the next model
print("\nAll listed models are done (or skipped).")

## 8. Compare all runs

In [ ]:
!python scripts/compare_results.py results

## 9. Download the results

In [ ]:
!zip -qr /content/results.zip results/
files.download("/content/results.zip")